In [2]:
%load_ext dotenv
%dotenv /home/aurora/.env
%matplotlib inline

In [3]:
import numpy as np
import pandas as pd
import datetime as dt
from google.cloud import bigquery
import re
import pymongo
import os
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
from flatten_json import flatten
#warnings.filterwarnings('ignore')

In [4]:
a = pd.read_csv('./123.csv')
b = a.transpose()
new_header = b.iloc[0]
b = b[1:]
b.columns = new_header
b.head()

Unnamed: 0,Android,iOS,nan,Months,Android Retention Rate,nan,nan,nan,nan,iOS Retention Rate,nan,nan,nan
Jul-19,"262,500","37,500",NaN,Year,FY19,FY20,FY21,FY22,NaN,FY19,FY20,FY21,FY22
Aug-19,"262,500","37,500",NaN,D30,10.0%,11.0%,12.5%,14.0%,NaN,12.0%,13.0%,14.5%,16.0%
Sep-19,"437,500","62,500",NaN,D60,8.0%,9.0%,10.5%,12.0%,NaN,10.0%,11.0%,12.5%,14.0%
Oct-19,"437,500","62,500",NaN,D90,5.0%,6.0%,7.5%,9.0%,NaN,8.0%,9.0%,10.5%,12.0%
Nov-19,"525,000","75,000",NaN,D120,4.0%,5.0%,6.5%,8.0%,NaN,6.0%,7.0%,8.5%,10.0%


In [5]:
c = b[['iOS']]
c['iOS'] = c['iOS'].str.replace(',','')
c['iOS'] = c['iOS'].astype(int)

/home/aurora/miniconda3/lib/python3.7/site-packages/ipykernel_launcher.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy
  
/home/aurora/miniconda3/lib/python3.7/site-packages/ipykernel_launcher.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy
  This is separate from the ipykernel package so we can avoid doing imports until


In [8]:
c.index.tolist()

['Jul-19',
 'Aug-19',
 'Sep-19',
 'Oct-19',
 'Nov-19',
 'Dec-19',
 'Jan-20',
 'Feb-20',
 'Mar-20',
 'Apr-20',
 'May-20',
 'Jun-20',
 'Jul-20',
 'Aug-20',
 'Sep-20',
 "Oct-Dec'20",
 "Jan-Mar'21",
 "Apr-Jun'21",
 "Jul-Sept'21",
 "Oct-Dec'21",
 "Jan-Mar'22",
 "Apr-Jun'22",
 "Jul-Sept'22",
 "Oct-Dec'22",
 "Jan-Mar'23"]

In [6]:
e = c.loc["Oct-Dec'20":]

split = e['iOS'].tolist()
split[:] = [x // 3 for x in split]
split = [item for item in split for i in range(3)]
x11 = pd.DataFrame({'iOS':split})

In [7]:
from datetime import timedelta, date
x1 = c.loc[:"Sep-20"]
y = pd.concat([x1,x11],ignore_index=True)

In [8]:
def daterange(date1, date2):
    for n in range(int ((date2 - date1).days)+1):
        yield date1 + timedelta(n)
date_list = []
start_dt = date(19,7,1)
end_dt = date(23,3,30)
for dt in daterange(start_dt, end_dt):
    date_list.append(dt.strftime("%Y-%m"))
    
xy = pd.DataFrame({'months': date_list})
xy.drop_duplicates('months',inplace = True)
xy.reset_index(drop=True, inplace=True)

In [9]:
z = pd.merge(y, xy , left_index=True,right_index = True)
z.set_index('months',inplace=True)

In [10]:
zero = []
for i in range(24):
    temp = []
    for j in range(i+1):
        temp.append(0)
    zero.append(temp)
anusers = z['iOS'].tolist()
columns = []
for i in range(24):
    temp1 = zero[i] + anusers
    temp1 = temp1[:45]
    columns.append(temp1)

In [22]:
dlist = []
for i in range(24):
    d30 = (a.iloc[9:13,(i+2)].tolist())
    d30 = [item for item in d30 for j in range(12)]
    d30 = d30[3:(3+45)]
    dlist.append(d30)

dlist1 = []
for i in range(24):
    t1 = zero[i] + dlist[i]
    t1 = t1[:45]
    dlist1.append(t1)

In [23]:
import itertools, pandas
c1 = pd.DataFrame((_ for _ in itertools.zip_longest(*columns)), columns=['c1', 'c2', 'c3','c4','c5','c6','c7','c8','c9','c10','c11','c12','c13','c14','c15','c16','c17','c18','c19','c20','c21','c22','c23','c24'])
#c1.loc[:,'Total'] = c1.sum(axis=1)

In [24]:
import itertools, pandas
c2 = pd.DataFrame((_ for _ in itertools.zip_longest(*dlist1)), columns=['c1', 'c2', 'c3','c4','c5','c6','c7','c8','c9','c10','c11','c12','c13','c14','c15','c16','c17','c18','c19','c20','c21','c22','c23','c24'])

c2 = c2.replace('%','',regex=True).astype('float')/100

In [25]:
c3 = pd.DataFrame(c1.values*c2.values, columns=c1.columns, index=c1.index)

c4 = c3.sum(axis=1)
c4 = c4.to_frame().reset_index()
c4.columns = ['index','totals']
c4 = c4[['totals']]
c4 = c4.set_index(z.index)

In [26]:
d = pd.merge(z, c4, left_index=True, right_index=True)
d = d.apply(pd.to_numeric)
d.head()

,iOS,totals
months,,
19-07,37500,0.0
19-08,37500,4500.0
19-09,62500,8250.0
19-10,62500,14250.0
19-11,75000,19000.0


In [27]:
e = d.sum(axis=1)
e = e.to_frame().reset_index()
e.columns = ['month','total number of active android users']
e = e.astype(int, errors='ignore')
e.set_index('month',inplace = True)
f = pd.merge(d, e, left_index=True,right_index = True)
f.columns = ['new users','retained users','total android users']
f

,new users,retained users,total android users
months,,,
19-07,37500,0.000,37500
19-08,37500,4500.000,42000
19-09,62500,8250.000,70750
19-10,62500,14250.000,76750
19-11,75000,19000.000,94000
19-12,75000,24375.000,99375
20-01,87500,28625.000,116125
20-02,87500,33500.000,121000
20-03,87500,37250.000,124750
